<a href="https://colab.research.google.com/github/Mufliha-Hafsyah/Machine-Learning/blob/main/Week06/Praktikum6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM PEMBELAJARAN MESIN**

**Nama**  : Mufliha Hafsyah Shahieza <br>
**NIM**   : 244107020147 <br>
**Kelas** : TI-3G

---

# **JS06 - ANN (Approximate Nearest Neighbors)**<hr>
# **Praktikum 6**<hr>

In [ ]:
!pip install faiss-cpu annoy hnswlib

  Using cached faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.8 kB)
  Using cached annoy-1.17.3.tar.gz (647 kB)
  Preparing metadata (setup.py) ... done
  Using cached hnswlib-0.8.0.tar.gz (36 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 29.1 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520036 sha256=916d7f7b164440db67989b9d5d8191a019e203599aeef3179da6702218454e1f
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=96ae11c57ec1c298c899a19b52c5695bae4bc29e1971f12ff1216a71a6753707
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Succes

In [ ]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------
# Load dataset
# -------------------
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/JOBSHEET ML/WEEK 6 ANN/songs_with_attributes_and_lyrics.csv')
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

# -------------------
# Exact Nearest Neighbor (brute-force)
# -------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------
# Annoy
# -------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------
# HNSW
# -------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------
# FAISS IVF
# -------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1],100,faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Exact NN done in 5678.140 s
Annoy done in 76.611 s
HNSW done in 365.666 s
FAISS IVF done in 989.694 s

Top-5 neighbors for first song:
Exact NN: [     0 394553 764272 837727 749223]
Annoy:    [0, 764272, 749223, 625227, 833164]
HNSW:     [     0 394553 764272 837727 749223]
FAISS:    [     0 394553 764272 837727 749223]


### **Hasil Praktikum 6: ANN pada Dataset Spotify (k=10, Euclidean, fitur distandarisasi)**

| Metode | Waktu total (detik) | Waktu total (menit) | Percepatan vs Exact NN | Top-5 tetangga lagu pertama | Cocok dengan Exact (top-5) |
|---|---|---|---|---|---|
| Exact NN (brute force) | 5678.140 | sekitar 94,6 | 1x (acuan) | [0, 394553, 764272, 837727, 749223] | 5/5 (acuan) |
| Annoy (10 tree) | 76.611 | sekitar 1,3 | sekitar 74x | [0, 764272, 749223, 625227, 833164] | 3/5 |
| HNSW (ef=200) | 365.666 | sekitar 6,1 | sekitar 15,5x | [0, 394553, 764272, 837727, 749223] | 5/5 |
| FAISS IVF (nlist=100, nprobe=10) | 989.694 | sekitar 16,5 | sekitar 5,7x | [0, 394553, 764272, 837727, 749223] | 5/5 |

**Analisis:**

1. **Kecepatan.** Exact NN butuh sekitar 94,6 menit karena membandingkan setiap lagu dengan seluruh lagu lain. Semua metode ANN jauh lebih cepat: Annoy sekitar 74 kali, HNSW sekitar 15,5 kali, dan FAISS IVF sekitar 5,7 kali lebih cepat.
2. **Akurasi (pada lagu pertama).** HNSW dan FAISS IVF menghasilkan 5 tetangga teratas yang identik dengan Exact NN, sedangkan Annoy hanya cocok 3 dari 5 (indeks 0, 764272, dan 749223; dua lainnya berbeda). Ini menunjukkan trade-off kecepatan vs akurasi: Annoy tercepat tetapi paling kurang akurat dengan 10 tree.
3. **Biaya yang dibayar tiap metode.** HNSW membutuhkan waktu build graf yang besar (sehingga total sekitar 6 menit) tetapi memberi akurasi tinggi. Annoy membangun indeks paling cepat. FAISS IVF dengan `nprobe=10` memeriksa 10 dari 100 cluster untuk setiap query sehingga total waktunya lebih lama dari HNSW dan Annoy pada data sebesar ini, tetapi hasilnya akurat.
4. **Keterbatasan.** Perbandingan akurasi di atas hanya berdasar satu lagu, sehingga belum bisa digeneralisasi. Waktu juga bergantung pada spesifikasi mesin Colab saat dijalankan.